In [43]:
import pandas as pd
import os
import numpy as np
from pyunicorn.timeseries import RecurrencePlot
from sklearn.preprocessing import MinMaxScaler


In [ ]:
# Sistema 1
#selected_channels_config = {
#    "mu": ["C2", "C1", "FCz", "Cz", "C3", "CP3"],

#    "beta": ["FC4", "C1", "Cz", "C1", "P3"],
    
#    "gamma": ["C3", "C2", "FC3", "CP3"]
#}


selected_channels_config = {
    "mu": ["C2", "C1", "FCz", "Cz", "C3", "CP3"],
    "beta": ["C2", "C1", "FCz", "Cz", "C3", "CP3"],
    "gamma": ["C2", "C1", "FCz", "Cz", "C3", "CP3"]
}



In [45]:

def recurrence_analysis(file_path, band="mu"):
    """
    Realiza análisis de recurrencia en datos EEG
    
    Args:
        file_path: Ruta al archivo CSV
        band: Banda de frecuencia ("mu", "beta", "gamma")
    
    Returns:
        DataFrame con métricas RQA y etiquetas binarias
    """
    try:
        # Leer datos
        data = pd.read_csv(file_path)
        
        # Verificar que existe la columna binaria
        if "Binaria" not in data.columns:
            print(f"Advertencia: No se encontró columna 'Binaria' en {file_path}")
            return None
            
        binary_signal = data["Binaria"].values
        
        # Seleccionar canales según la banda
        selected_channels = selected_channels_config[band]
        selected_columns = [f"{band}_" + ch for ch in selected_channels]
        
        # Verificar que existen las columnas necesarias
        missing_columns = [col for col in selected_columns if col not in data.columns]
        if missing_columns:
            print(f"Advertencia: Columnas faltantes en {file_path}: {missing_columns}")
            return None
        
        eeg_data = data[selected_columns].values
        
        eeg_data_normalized = np.array(eeg_data)
        
        # Parámetros de ventana deslizante
        win_len = 60
        overlap = 0.50
        step = max(1, int(win_len * (1 - overlap)))
        
        # Listas para almacenar resultados
        rqa_metrics = {
            "recurrence_rate": [],
            "determinism": [],
            "trapping_time": [],
            "diag_entropy": [],
            "vert_entropy": [],
            "laminarity": [],
            
        }
        
        binary_labels = []
        
        # Calcular threshold dinámico basado en toda la serie
        overall_std = np.std(eeg_data_normalized)
        threshold = overall_std * 0.8
        if band == "mu":
            threshold = 0.08
        elif band == "beta":
            threshold = 0.12
        elif band == "gamma":
            threshold = 0.12

        print(f"Threshold calculado: {threshold:.4f}")
        # Análisis por ventanas
        for i in range(0, len(binary_signal) - win_len + 1, step):
            window_data = eeg_data_normalized[i:i + win_len, :]
            binary_labels.append(binary_signal[i])
            
            try:
                # Crear RecurrencePlot usando pyunicorn
                rp = RecurrencePlot(
                    window_data,
                    #recurrence_rate= 0.09,  # Ajustar según sea necesario
                    threshold=threshold,
                    
                    adaptive_neighborhood_size = 20,
                    dim = 10, 
                    epsilon='distance',
                    metric="euclidean",
                )
                
                # Extraer métricas RQA
                rqa_metrics["recurrence_rate"].append(rp.recurrence_rate())
                rqa_metrics["determinism"].append(rp.determinism())
                rqa_metrics["trapping_time"].append(rp.trapping_time())
                rqa_metrics["diag_entropy"].append(rp.diag_entropy())
                rqa_metrics["vert_entropy"].append(rp.vert_entropy())
                rqa_metrics["laminarity"].append(rp.laminarity())
                
                
            except Exception as e:
                print(f"Error en ventana {i}: {e}")
                # Agregar valores NaN si falla el análisis
                for key in rqa_metrics.keys():
                    rqa_metrics[key].append(np.nan)
        
        # Crear DataFrame resultado
        df_result = pd.DataFrame(rqa_metrics)
        df_result["Binaria"] = binary_labels
        df_result["banda"] = band
        df_result["archivo"] = os.path.basename(file_path)
        
        return df_result
        
    except Exception as e:
        print(f"Error procesando {file_path}: {e}")
        return None


In [46]:

def read_csvs(folder_path, band="mu"):
    """
    Lee y procesa todos los archivos CSV en una carpeta
    
    Args:
        folder_path: Ruta a la carpeta con archivos CSV
        band: Banda de frecuencia a analizar
    
    Returns:
        Tupla con DataFrames (temp0_reg, temp5_reg, temp10_reg)
    """
    if not os.path.exists(folder_path):
        print(f"La carpeta {folder_path} no existe")
        return None, None, None
    
    files = os.listdir(folder_path)
    csv_files = [file for file in files if file.endswith('.csv')]
    
    if not csv_files:
        print(f"No se encontraron archivos CSV en {folder_path}")
        return None, None, None
    
    # Inicializar variables de resultado
    temp0_reg = None
    temp5_reg = None
    temp10_reg = None
    
    # Mapeo de archivos
    file_mapping = {
        "pasivoPre.csv": ("temp0_reg", "0% Torque"),
        "5deTorquePre.csv": ("temp5_reg", "5% Torque"),
        "10deTorquePre.csv": ("temp10_reg", "10% Torque")
    }
    
    for csv_file in csv_files:
        if csv_file in file_mapping:
            var_name, condition = file_mapping[csv_file]
            file_path = os.path.join(folder_path, csv_file)
            
            print(f"Procesando {csv_file} ({condition})...")
            
            try:
                result = recurrence_analysis(file_path, band)
                if result is not None:
                    result["condicion"] = condition
                    
                    if var_name == "temp0_reg":
                        temp0_reg = result
                    elif var_name == "temp5_reg":
                        temp5_reg = result
                    elif var_name == "temp10_reg":
                        temp10_reg = result
                        
                    print(f"✓ {csv_file} procesado exitosamente ({len(result)} ventanas)")
                else:
                    print(f"✗ Error procesando {csv_file}")
                    
            except Exception as e:
                error_msg = f"Error procesando {csv_file}: {e}\n"
                print(error_msg)
                with open("errors.log", "a", encoding='utf-8') as error_file:
                    error_file.write(f"{file_path}: {error_msg}")
    
    return temp0_reg, temp5_reg, temp10_reg

In [47]:

def process_all_subjects(root_folder_path, band="mu", save_results=True):
    """
    Procesa todos los sujetos en el directorio raíz
    
    Args:
        root_folder_path: Ruta al directorio raíz
        band: Banda de frecuencia a analizar
        save_results: Si guardar los resultados en CSV
    
    Returns:
        DataFrames consolidados por condición
    """
    if not os.path.exists(root_folder_path):
        print(f"La ruta {root_folder_path} no existe")
        return None, None, None
    
    # Listas para consolidar todos los datos
    all_carga_0 = []
    all_carga_5 = []
    all_carga_10 = []
    
    # Obtener carpetas de sujetos
    items_in_root = os.listdir(root_folder_path)
    sub_folders = [item for item in items_in_root 
                   if os.path.isdir(os.path.join(root_folder_path, item))]
    
    print(f"Encontrados {len(sub_folders)} sujetos para procesar")
    
    for subject_folder in sub_folders:
        subject_path = os.path.join(root_folder_path, subject_folder)
        print(f"\n--- Procesando sujeto: {subject_folder} ---")
        
        # Obtener sesiones del sujeto
        items_in_subject = os.listdir(subject_path)
        session_folders = [item for item in items_in_subject 
                          if os.path.isdir(os.path.join(subject_path, item))]
        
        for session_folder in session_folders:
            session_path = os.path.join(subject_path, session_folder)
            print(f"  Procesando sesión: {session_folder}")
            
            try:
                # Procesar archivos de la sesión
                temp0, temp5, temp10 = read_csvs(session_path, band)
                
                # Agregar información de sujeto y sesión
                for temp_data, temp_list in [(temp0, all_carga_0), 
                                           (temp5, all_carga_5), 
                                           (temp10, all_carga_10)]:
                    if temp_data is not None:
                        temp_data["sujeto"] = subject_folder
                        temp_data["sesion"] = session_folder
                        temp_list.append(temp_data)
                
            except Exception as e:
                error_message = f"Error procesando {session_path}: {e}\n"
                print(error_message)
                with open("errors.log", "a", encoding='utf-8') as error_file:
                    error_file.write(error_message)
                continue
    
    # Consolidar resultados
    final_carga_0 = pd.concat(all_carga_0, ignore_index=True) if all_carga_0 else None
    final_carga_5 = pd.concat(all_carga_5, ignore_index=True) if all_carga_5 else None
    final_carga_10 = pd.concat(all_carga_10, ignore_index=True) if all_carga_10 else None
    
    # Guardar resultados si se solicita
    if save_results:
        output_folder = f"resultados_rqa_{band}"
        os.makedirs(output_folder, exist_ok=True)
        
        for df, name in [(final_carga_0, "carga_0"), 
                        (final_carga_5, "carga_5"), 
                        (final_carga_10, "carga_10")]:
            if df is not None:
                output_path = os.path.join(output_folder, f"{name}_{band}.csv")
                df.to_csv(output_path, index=False)
                print(f"Guardado: {output_path} ({len(df)} registros)")
    
    return final_carga_0, final_carga_5, final_carga_10

In [48]:
root_folder_path = r"E:\Pruebas%20BCI\Completos10Hz"
#'d:\Pruebas%20BCI\Completos10Hz'
    
# Procesar para cada banda
bandas = ["mu", "beta", "gamma"]

for banda in bandas:
    print(f"\n{'='*50}")
    print(f"PROCESANDO BANDA: {banda.upper()}")
    print(f"{'='*50}")
    
    try:
        carga_0, carga_5, carga_10 = process_all_subjects(
            root_folder_path, 
            band=banda, 
            save_results=True
        )
        
        # Mostrar resumen
        print(f"\nResumen para banda {banda}:")
        for df, name in [(carga_0, "Carga 0%"), 
                        (carga_5, "Carga 5%"), 
                        (carga_10, "Carga 10%")]:
            if df is not None:
                print(f"  {name}: {len(df)} ventanas de {df['sujeto'].nunique()} sujetos")
            else:
                print(f"  {name}: Sin datos")
                
    except Exception as e:
        print(f"Error procesando banda {banda}: {e}")

print("\n¡Procesamiento completado!")


PROCESANDO BANDA: MU
Encontrados 14 sujetos para procesar

--- Procesando sujeto: AlejandoPayan ---
  Procesando sesión: S1
Procesando 10deTorquePre.csv (10% Torque)...
Threshold calculado: 0.0800
Calculating recurrence plot at fixed threshold...
Calculating the euclidean distance matrix...
Calculating recurrence plot at fixed threshold...
Calculating the euclidean distance matrix...
Calculating recurrence plot at fixed threshold...
Calculating the euclidean distance matrix...
Calculating recurrence plot at fixed threshold...
Calculating the euclidean distance matrix...
Calculating recurrence plot at fixed threshold...
Calculating the euclidean distance matrix...
Calculating recurrence plot at fixed threshold...
Calculating the euclidean distance matrix...
Calculating recurrence plot at fixed threshold...
Calculating the euclidean distance matrix...
Calculating recurrence plot at fixed threshold...
Calculating the euclidean distance matrix...
Calculating recurrence plot at fixed thres